# Notebook 07: Test-Time Scaling

**Time:** 25 minutes  
**Prerequisites:** Notebook 06 complete  
**Goal:** Understand and experiment with computation strategies that improve LLM quality at inference time

> 💡 **Key insight from Week 2 lecture:** We can improve model output quality without changing model weights —
> by allocating more compute *at inference time*. This is the core idea behind O1 and O3.

## Three Techniques

1. **Chain-of-Thought (CoT)** — encourage step-by-step reasoning (no extra compute cost)
2. **Extended Thinking** — model reasons in a private "scratchpad" before answering (Path A/C)
3. **Quantization** — reduce memory to allow larger models or faster inference

In [1]:
import os, sys, json, time
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir    = os.path.join('..', 'outputs')
thinking_dir   = os.path.join(outputs_dir, 'thinking_traces')
os.makedirs(thinking_dir, exist_ok=True)

print("✅ Setup complete — ready for Notebook 07")

/Users/scottlai/Documents/inferenceAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
✅ Setup complete — ready for Notebook 07


---

## Part 1: Chain-of-Thought Prompting

CoT doesn't add compute — it *reallocates* compute from the next-token prediction step to explicit reasoning tokens. The model expresses its intermediate reasoning in the output, which forces more structured thinking.

In [2]:
print("=" * 65)
print("🧪 Experiment 1: Chain-of-Thought Comparison")
print("=" * 65)
print()

# A problem that benefits from step-by-step reasoning
problem = (
    "A dataset has 10,000 documents. After language filtering, 85% remain. "
    "After MinHash deduplication (removing 30% of remaining), how many documents "
    "are left? Then 5% of what remains contains PII and is removed. "
    "What percentage of the original dataset survives the full pipeline?"
)

styles = [
    ("Direct",      problem,                                      None),
    ("Step-by-step", problem + "\n\nSolve step by step.",         None),
    ("Few-shot CoT", 
     "Example: 100 items → filter 20% → 80 left. → remove 50% → 40 left.\n"
     "Now solve:\n" + problem,
     "You are a precise data science calculator."),
]

results_exp1 = []
for style_name, prompt, system in styles:
    start = time.time()
    resp = client.generate(
        prompt=prompt, system=system,
        max_tokens=400, temperature=0.0   # deterministic for fair comparison
    )
    elapsed = time.time() - start

    if "error" not in resp:
        tracker.add_call(resp)
        out_tokens = resp['usage']['output_tokens']
        results_exp1.append((style_name, resp['content'], out_tokens, elapsed))
        print(f"[{style_name}] {out_tokens} output tokens in {elapsed:.1f}s")
        print(f"  {resp['content'][:200]}...")
        print()

print()
print("💡 Notice how more tokens ≠ more accuracy. The STRUCTURE of reasoning matters.")

🧪 Experiment 1: Chain-of-Thought Comparison

[Direct] 400 output tokens in 6.0s
  ## Step-by-Step Pipeline Calculation

### Stage 1: Language Filtering
$$10{,}000 \times 0.85 = 8{,}500 \text{ documents}$$

### Stage 2: MinHash Deduplication (remove 30% of remaining)
$$8{,}500 \time...

[Step-by-step] 400 output tokens in 6.2s
  # Document Processing Pipeline - Step by Step Solution

## Starting Point
**Initial dataset:** 10,000 documents

---

## Step 1: Language Filtering

> 85% of documents remain after filtering

$$10,000...

[Few-shot CoT] 379 output tokens in 5.7s
  ## Step-by-Step Solution

---

### Step 1: Language Filtering
$$10{,}000 \times 0.85 = 8{,}500 \text{ documents}$$

---

### Step 2: MinHash Deduplication (removing 30% of remaining)
$$8{,}500 \times ...


💡 Notice how more tokens ≠ more accuracy. The STRUCTURE of reasoning matters.


### 🎯 TODO 1: Design a CoT Problem for Your Domain

In [3]:
# TODO 1: Create a problem from your project domain that benefits from CoT
#         Test all three styles and compare the quality of answers.

my_problem = """
[YOUR PROBLEM HERE]

Examples of good CoT problems:
  - Multi-step calculations (data pipeline, cost estimation, scheduling)
  - Logic puzzles with constraints
  - Clinical reasoning: "Given symptoms A, B, C — what is the most likely diagnosis?"
  - Code debugging: "This function fails when input is X. Why? How to fix?"
"""

print("=" * 65)
print("🎯 TODO 1: CoT on My Domain Problem")
print("=" * 65)
print()

for style_name, suffix in [("Direct", ""), ("CoT", "\n\nSolve step by step.")]:
    resp = client.generate(
        prompt=my_problem + suffix,
        max_tokens=500, temperature=0.0
    )
    if "error" not in resp:
        tracker.add_call(resp)
        print(f"[{style_name}] {resp['usage']['output_tokens']} tokens")
        print(resp['content'])
        print()

todo1_reflection = """
[YOUR REFLECTION HERE]

- Which style produced the more accurate answer? How do you know?
- Did CoT use more tokens? Was that extra cost worth it?
- For what types of problems does CoT help the most in your domain?
"""
print(todo1_reflection)

🎯 TODO 1: CoT on My Domain Problem

[Direct] 155 tokens
It looks like your message came through empty — the **[YOUR PROBLEM HERE]** placeholder wasn't filled in! 😊

Please share the problem you'd like help with, and I'll walk through it step by step. I'm ready to help with things like:

- 🔢 **Multi-step calculations** (costs, schedules, data pipelines)
- 🧩 **Logic puzzles** with constraints
- 🏥 **Clinical reasoning** (symptoms → diagnosis)
- 💻 **Code debugging** (finding and fixing bugs)
- 📊 **Data analysis or estimation problems**

**Just paste your problem and I'll get started!**

[CoT] 148 tokens
It looks like your message came through as a template placeholder — **"[YOUR PROBLEM HERE]"** — without an actual problem attached! 😊

Could you share the problem you'd like me to solve? I'm ready to work through it **step by step**, whether it's:

- 🔢 A **math or calculation** problem
- 🧩 A **logic puzzle**
- 🏥 A **clinical/diagnostic** scenario
- 💻 A **code debugging** challenge
- 📊 A **d

---

## Part 2: Extended Thinking (Path A/C — Claude API)

Claude's **extended thinking** lets the model reason in a private scratchpad before producing its final answer. This is similar to OpenAI's o1/o3 reasoning tokens.

Unlike CoT (where reasoning is in the output), extended thinking:
- Reasoning is hidden from the user by default (but accessible via the API)
- Budget controls how many tokens to spend on reasoning
- Particularly effective for math, logic, multi-step problems

In [4]:
print("=" * 65)
print("🧪 Experiment 2: Extended Thinking")
print("=" * 65)
print()

logic_puzzle = """
Five ML engineers (Alice, Bob, Carol, Dave, Eve) are assigned to one of five tasks:
data collection, cleaning, tokenization, pretraining, and evaluation.

Constraints:
1. Alice cannot do pretraining (insufficient GPU budget)
2. Bob is doing either data collection or cleaning
3. Carol is doing tokenization or evaluation
4. Dave and Eve are not doing adjacent pipeline stages
   (collection→cleaning→tokenization→pretraining→evaluation)
5. Eve is not doing data collection

Who does what? Show all your reasoning.
"""

if config.PATH in ["A", "C"]:
    print("Running extended thinking (budget_tokens=2000)...")
    start = time.time()
    
    resp_thinking = client.generate_with_thinking(
        prompt=logic_puzzle,
        budget_tokens=2000,
        max_tokens=4096
    )
    elapsed = time.time() - start

    if "error" not in resp_thinking:
        tracker.add_call(resp_thinking)

        print(f"✅ Response in {elapsed:.1f}s")
        print(f"   Tokens: {resp_thinking['usage']['input_tokens']}in / {resp_thinking['usage']['output_tokens']}out")
        print()
        print("--- THINKING (internal reasoning) ---")
        thinking = resp_thinking.get('thinking', '')
        print(thinking[:600] + ('...' if len(thinking) > 600 else ''))
        print()
        print("--- FINAL ANSWER ---")
        print(resp_thinking['content'])

        # Save thinking trace
        trace_path = os.path.join(thinking_dir, 'logic_puzzle_thinking.json')
        with open(trace_path, 'w') as f:
            json.dump({
                "prompt": logic_puzzle,
                "thinking": resp_thinking.get('thinking', ''),
                "answer": resp_thinking['content'],
                "budget_tokens": 2000,
                "usage": resp_thinking['usage'],
            }, f, indent=2)
        print(f"\n✅ Thinking trace saved: {trace_path}")
    else:
        print(f"❌ Error: {resp_thinking['error']}")

else:
    print("Path B (Ollama) — using multi-turn scratchpad simulation instead")
    print()
    # Simulate extended thinking with explicit scratchpad
    scratchpad_prompt = (
        "Before answering, work through this step by step in a <thinking> block. "
        "Then give your final answer after </thinking>.\n\n" + logic_puzzle
    )
    resp_scratchpad = client.generate(prompt=scratchpad_prompt, max_tokens=800, temperature=0.0)
    if "error" not in resp_scratchpad:
        tracker.add_call(resp_scratchpad)
        print(format_response(resp_scratchpad, verbose=True))

🧪 Experiment 2: Extended Thinking

Running extended thinking (budget_tokens=2000)...
✅ Response in 58.9s
   Tokens: 162in / 4096out

--- THINKING (internal reasoning) ---
Let me work through this systematically.

Tasks: data collection (1), cleaning (2), tokenization (3), pretraining (4), evaluation (5)
People: Alice, Bob, Carol, Dave, Eve

Constraints:
1. Alice ≠ pretraining
2. Bob = data collection OR cleaning
3. Carol = tokenization OR evaluation
4. Dave and Eve not adjacent (adjacent pairs: 1-2, 2-3, 3-4, 4-5)
5. Eve ≠ data collection

From constraint 2: Bob does collection or cleaning (stages 1 or 2)


From constraint 3: Carol does tokenization or evaluation (stages 3 or 5)
From constraint 5: Eve can't take stage 1

So Bob occupies either stage 1 or 2, Car...

--- FINAL ANSWER ---


✅ Thinking trace saved: ../outputs/thinking_traces/logic_puzzle_thinking.json


### 🎯 TODO 2: Your Project Domain + Extended Thinking Budget Comparison

In [5]:
# TODO 2: Run extended thinking on a problem from your project domain
#         Compare budget_tokens=500 vs budget_tokens=3000

my_thinking_problem = """
[YOUR PROBLEM HERE — should benefit from deep reasoning]

Good candidates:
  - A complex design decision: "Should I use RAG or fine-tuning for [use case]?"
  - A tricky data analysis question in your domain
  - An ethical dilemma about AI deployment in your field
  - A multi-constraint optimization problem
"""

print("=" * 65)
print("🎯 TODO 2: Extended Thinking Budget Comparison")
print("=" * 65)
print()

budgets = [500, 3000]
budget_results = []

for budget in budgets:
    print(f"\n--- Budget: {budget} tokens ---")
    if config.PATH in ["A", "C"]:
        resp = client.generate_with_thinking(
            prompt=my_thinking_problem,
            budget_tokens=budget,
            max_tokens=budget + 2048
        )
    else:
        # Path B fallback
        resp = client.generate(
            prompt=f"Think step by step with {budget} words of reasoning.\n{my_thinking_problem}",
            max_tokens=budget
        )

    if "error" not in resp:
        tracker.add_call(resp)
        answer = resp['content']
        thinking_len = len(resp.get('thinking', ''))
        budget_results.append((budget, answer, thinking_len))

        print(f"  Thinking length: {thinking_len} chars")
        print(f"  Answer preview:  {answer[:300]}...")

        # Save trace
        trace_path = os.path.join(thinking_dir, f'project_problem_budget{budget}.json')
        with open(trace_path, 'w') as f:
            json.dump({
                "prompt": my_thinking_problem,
                "budget_tokens": budget,
                "thinking": resp.get('thinking', ''),
                "answer": answer,
                "usage": resp['usage'],
            }, f, indent=2)
        print(f"  ✅ Saved: {trace_path}")

todo2_reflection = """
[YOUR REFLECTION HERE]

- How did the answer quality change between budget=500 and budget=3000?
- Was the quality improvement worth the extra cost? (More output tokens = more cost)
- For your production use case, what budget_tokens setting would you choose?
- When would you use extended thinking vs. standard CoT?
"""
print()
print(todo2_reflection)

🎯 TODO 2: Extended Thinking Budget Comparison


--- Budget: 500 tokens ---

--- Budget: 3000 tokens ---
  Thinking length: 230 chars
  Answer preview:  # No Problem Detected Yet

It looks like you sent the **template instructions** rather than an actual problem to solve.

The placeholder `[YOUR PROBLEM HERE]` wasn't filled in.

---

## Here's What To Do

**Paste or describe your actual problem.** Good examples:

| Type | Example |
|------|---------...
  ✅ Saved: ../outputs/thinking_traces/project_problem_budget3000.json


[YOUR REFLECTION HERE]

- How did the answer quality change between budget=500 and budget=3000?
- Was the quality improvement worth the extra cost? (More output tokens = more cost)
- For your production use case, what budget_tokens setting would you choose?
- When would you use extended thinking vs. standard CoT?



---

## Part 3: Quantization — More Model for Less Memory

In [6]:
print("=" * 65)
print("🧪 Experiment 3: Float32 vs Float16 Memory & Speed")
print("=" * 65)
print()

try:
    import torch
    
    # Model size: realistic embedding layer (vocab=50k, dim=512)
    vocab_size = 50_000
    d_model    = 512
    
    # Float32 (standard)
    embedding_f32 = torch.nn.Embedding(vocab_size, d_model)  # 4 bytes per param
    params_f32 = embedding_f32.weight.numel()
    bytes_f32  = params_f32 * 4
    
    # Float16 (half precision)
    embedding_f16 = embedding_f32.half()                      # 2 bytes per param
    bytes_f16  = params_f32 * 2

    print(f"Embedding layer: vocab={vocab_size}, dim={d_model}")
    print(f"  Parameters: {params_f32:,}")
    print()
    print(f"  float32 size: {bytes_f32 / 1024**2:.1f} MB")
    print(f"  float16 size: {bytes_f16 / 1024**2:.1f} MB")
    print(f"  Memory saved: {(1 - bytes_f16/bytes_f32)*100:.0f}%")
    print()

    # Speed benchmark
    x = torch.randint(0, vocab_size, (32, 512))  # batch=32, seq_len=512

    # Warm up
    for _ in range(3):
        _ = embedding_f32(x)

    # Float32 timing
    start = time.time()
    for _ in range(100):
        _ = embedding_f32(x)
    t32 = time.time() - start

    # Float16 timing
    start = time.time()
    for _ in range(100):
        _ = embedding_f16(x)
    t16 = time.time() - start

    speedup = t32 / t16
    print(f"  float32 time (100 batches): {t32:.3f}s")
    print(f"  float16 time (100 batches): {t16:.3f}s")
    print(f"  Speedup: {speedup:.1f}x")
    print()

    # Scale to a real model
    print("Scaling to full model sizes:")
    print()
    for model_name, params_B in [("LLaMA 7B", 7), ("LLaMA 13B", 13), ("LLaMA 70B", 70)]:
        f32_gb = params_B * 4
        f16_gb = params_B * 2
        int8_gb = params_B * 1
        print(f"  {model_name:<12}  float32: {f32_gb}GB  float16: {f16_gb}GB  int8: {int8_gb}GB")

    print()
    print("💡 A consumer A100 (80GB) can run LLaMA 70B in float16 but NOT float32.")
    print("   QLoRA (4-bit quantization) fits 70B in a single 40GB A100!")

except ImportError:
    print("⚠️  PyTorch not installed — skipping experiment")

todo3_reflection = """
[YOUR REFLECTION HERE]

- What speedup did you observe for float16 vs float32?
- For qwen3.5:27b (your local model), approximately how much GPU memory does it need
  in float16? Would it fit on a consumer GPU (24GB)?
- What's the trade-off between int8 quantization vs float16?
- How does quantization relate to test-time scaling? (Hint: same GPU budget = ?)
"""
print()
print(todo3_reflection)

🧪 Experiment 3: Float32 vs Float16 Memory & Speed

Embedding layer: vocab=50000, dim=512
  Parameters: 25,600,000

  float32 size: 97.7 MB
  float16 size: 48.8 MB
  Memory saved: 50%

  float32 time (100 batches): 0.017s
  float16 time (100 batches): 0.015s
  Speedup: 1.2x

Scaling to full model sizes:

  LLaMA 7B      float32: 28GB  float16: 14GB  int8: 7GB
  LLaMA 13B     float32: 52GB  float16: 26GB  int8: 13GB
  LLaMA 70B     float32: 280GB  float16: 140GB  int8: 70GB

💡 A consumer A100 (80GB) can run LLaMA 70B in float16 but NOT float32.
   QLoRA (4-bit quantization) fits 70B in a single 40GB A100!


[YOUR REFLECTION HERE]

- What speedup did you observe for float16 vs float32?
- For qwen3.5:27b (your local model), approximately how much GPU memory does it need
  in float16? Would it fit on a consumer GPU (24GB)?
- What's the trade-off between int8 quantization vs float16?
- How does quantization relate to test-time scaling? (Hint: same GPU budget = ?)



## Summary & Reflection

In [7]:
full_reflection = f"""
### Chain-of-Thought Experiment (TODO 1)

My problem domain: {my_problem[:100].strip() if 'my_problem' in dir() else '[not set]'}

{todo1_reflection.strip()}

---

### Extended Thinking Budget Comparison (TODO 2)

Budget results: {[(b, len(a)) for b, a, _ in budget_results] if 'budget_results' in dir() else 'N/A'}

{todo2_reflection.strip()}

---

### Quantization Speedup (TODO 3 — Experiment 3)

{todo3_reflection.strip()}
"""

reflection_file = append_to_reflection(
    notebook="07",
    section_title="Test-Time Scaling Experiments",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)
print(f"✅ Reflection saved: {reflection_file}")
print(f"✅ Thinking traces saved to: {thinking_dir}")
print()
tracker.report()

✅ Reflection saved: ../outputs/homework_reflection.md
✅ Thinking traces saved to: ../outputs/thinking_traces

💰 API COST REPORT
Total API calls:     7
Total input tokens:  728
Total output tokens: 5,893
Total cost:          $0.0906

Last 5 calls:
  1. [23:25:58] sonnet — 114in/379out — $0.0060
  2. [23:26:24] sonnet — 90in/155out — $0.0026
  3. [23:26:28] sonnet — 97in/148out — $0.0025
  4. [23:27:47] sonnet — 162in/4096out — $0.0619
  5. [23:29:31] sonnet — 114in/315out — $0.0051


## ✅ Notebook 07 Complete!

**What you accomplished:**
- ✅ Compared direct vs CoT vs few-shot CoT on a multi-step problem
- ✅ Used Claude's extended thinking API and inspected the thinking trace
- ✅ Measured float32 vs float16 memory and speed difference
- ✅ Saved thinking traces to `outputs/thinking_traces/`

**Key concepts:**
- CoT = more reasoning tokens at no extra per-token cost, better structured answers
- Extended thinking = private reasoning scratchpad, more powerful than CoT for hard problems
- Quantization = same accuracy, 2–4× less memory, enables running larger models locally
- Test-time scaling: allocate more compute at inference → better output without retraining

**Next:** Open **Notebook 08: Project Integration** 🚀